这是 Qt 最核心的设计之一，让我从底层机制讲清楚。

## Qt 对象树机制

Qt 内部维护了一棵**对象树**。当你调用 `super().__init__(parent)` 时，Qt 在 `QObject` 基类的构造函数里做了这件事（C++ 层面的伪代码）：

```cpp
QObject::QObject(QObject* parent) {
    if (parent) {
        parent->children.append(this);  // 把自己加入父对象的 children 列表
    }
}

QObject::~QObject() {
    for (auto child : children) {
        delete child;  // 析构时，遍历并删除所有子对象
    }
}
```

## 图示流程

**创建时：**

```
model = TodoTableModel(window)
         │
         └──→ super().__init__(window)
                  │
                  └──→ QObject::QObject(window)
                           │
                           └──→ window.children.append(model)  // 注册到孩子列表
```

**销毁时（window 被关闭）：**

```
~QMainWindow()
  └──→ ~QObject()
           │
           └──→ 遍历 children 列表:
                    delete table_view     ✓
                    delete model          ✓  ← 自动销毁！
                    delete layout         ✓
                    ...
```

## 为什么这样设计？

**C++ 中手动管理内存很痛苦：**

```cpp
// 没有对象树——每次都要手动 delete
auto* window = new QMainWindow();
auto* model  = new TodoTableModel(window);
auto* view   = new QTableView();

// ... 后面必须记住：
delete view;
delete model;
delete window;
// 如果顺序错了或漏了，就是内存泄漏
```

Qt 用对象树解决了这个问题：**只管创建，不用管释放**。

## 在 Python 中的意义

Python 有垃圾回收，看起来不需要这个。但 Qt 的 C++ 对象不由 Python GC 管理——如果你不传 parent，C++ 层的 QObject 可能已经被释放了，而 Python 层的包装对象还悬在那里，导致崩溃（Segmentation Fault）。

所以结论是：**`parent` 参数本质是告诉 Qt "我归你管了"**，Qt 把当前对象加入父对象的 children 列表，父对象析构时递归删除所有孩子。这是 Qt 自动内存管理的核心机制。